# Noah Miller 

**Cluster 1 Stacking Model**

**Group 2** | Cluster 1: 1407 companies, 35 bankrupt (2.49% rate)

Note: moderate class imbalance with enough positive samples for reliable CV estimates.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    ExtraTreesClassifier, AdaBoostClassifier, StackingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.feature_selection import mutual_info_classif
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
CLUSTER_DIR = os.path.join(DATA_DIR, 'clusters')
MODEL_DIR = os.path.join(PROJECT_DIR, 'models')
SUBGROUP_MODEL_DIR = os.path.join(MODEL_DIR, 'subgroup_models')
os.makedirs(SUBGROUP_MODEL_DIR, exist_ok=True)

In [2]:
def custom_accuracy(y_true, y_pred):
    """Project metric: acc = TT / (TT + TF) = recall of bankrupt class.
    TT = correctly predicted bankrupt, TF = bankrupt predicted as non-bankrupt."""
    tt = ((y_true == 1) & (y_pred == 1)).sum()
    tf = ((y_true == 1) & (y_pred == 0)).sum()
    if tt + tf == 0:
        return 0.0
    return tt / (tt + tf)

def sparsity_check(y_pred, threshold=0.20):
    """Check if < 20% of predictions are bankrupt."""
    rate = y_pred.mean()
    return rate < threshold, rate

def evaluate_model(y_true, y_pred, label=''):
    """Print evaluation metrics."""
    acc = custom_accuracy(y_true, y_pred)
    ok, rate = sparsity_check(y_pred)
    cm = confusion_matrix(y_true, y_pred)
    print(f'--- {label} ---')
    print(f'Custom Accuracy (TT/(TT+TF)): {acc:.4f}')
    print(f'Sparsity: {rate:.4f} ({"PASS" if ok else "FAIL"})')
    print(f'Confusion Matrix:\n{cm}')
    print(f'Classification Report:\n{classification_report(y_true, y_pred, zero_division=0)}')
    return acc, rate

---
## Load & Inspect Data

In [3]:
# Load cluster 1 data
df1 = pd.read_csv(os.path.join(CLUSTER_DIR, 'cluster_1.csv'))
y1 = df1['Bankrupt?'].values
X1 = df1.drop(columns=['Bankrupt?']).values
feature_names_1 = df1.drop(columns=['Bankrupt?']).columns.tolist()

print(f'Cluster 1: {X1.shape[0]} samples, {X1.shape[1]} features')
print(f'Bankrupt: {y1.sum()} ({y1.mean():.4f})')

Cluster 1: 1407 samples, 95 features
Bankrupt: 35 (0.0249)


## Feature Selection

In [ ]:
# Feature selection using mutual information
mi_scores_1 = mutual_info_classif(X1, y1, random_state=RANDOM_STATE)
mi_series_1 = pd.Series(mi_scores_1, index=feature_names_1).sort_values(ascending=False)

N_FEATURES_1 = 3
top_features_1 = mi_series_1.head(N_FEATURES_1).index.tolist()
X1_sel = df1[top_features_1].values

print(f'Selected {N_FEATURES_1} features for Cluster 1:')
for i, (feat, score) in enumerate(mi_series_1.head(N_FEATURES_1).items()):
    print(f'  {i+1:2d}. {feat[:55]:55s}  MI = {score:.4f}')

Selected 10 features for Cluster 1:
   1. Borrowing dependency                                     MI = 0.0234
   2. Equity to Liability                                      MI = 0.0184
   3. Total debt/Total net worth                               MI = 0.0182
   4. Liability to Equity                                      MI = 0.0182
   5. Net worth/Assets                                         MI = 0.0180
   6. Current Liability to Assets                              MI = 0.0173
   7. Debt ratio %                                             MI = 0.0172
   8. Working capitcal Turnover Rate                           MI = 0.0170
   9. Retained Earnings to Total Assets                        MI = 0.0148
  10. Per Share Net profit before tax (Yuan ¥)                 MI = 0.0146


## Stacking Model Definition

In [ ]:
BEST_THRESH_1 = 0.47

print(f'Using {N_FEATURES_1} features, threshold={BEST_THRESH_1:.2f}')

base_estimators_1 = [
    ('rf', RandomForestClassifier(
        n_estimators=100, max_depth=4, min_samples_leaf=5,
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
    ('gb', GradientBoostingClassifier(
        n_estimators=80, max_depth=3, learning_rate=0.05,
        min_samples_leaf=10, random_state=RANDOM_STATE)),
    ('et', ExtraTreesClassifier(
        n_estimators=100, max_depth=4, min_samples_leaf=5,
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
]

meta_learner_1 = LogisticRegression(
    class_weight='balanced', C=0.1, max_iter=1000, random_state=RANDOM_STATE)

stacker_1 = StackingClassifier(
    estimators=base_estimators_1,
    final_estimator=meta_learner_1,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    stack_method='predict_proba',
    n_jobs=-1
)

print('Stacking classifier: 3 base models + regularized LR meta-learner.')

Using 3 features, threshold=0.47
Stacking classifier: 3 base models + regularized LR meta-learner.


## Cross-Validation

In [8]:
# Cross-validate with SMOTE + probability threshold tuning
cv1 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
y1_cv_probas = np.zeros(len(y1))

for fold, (train_idx, val_idx) in enumerate(cv1.split(X1_sel, y1)):
    X_train_cv, X_val_cv = X1_sel[train_idx], X1_sel[val_idx]
    y_train_cv, y_val_cv = y1[train_idx], y1[val_idx]
    
    # SMOTE — moderate oversampling
    n_pos = int(y_train_cv.sum())
    smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=min(5, n_pos - 1),
                  sampling_strategy=min(0.3, 1.0))
    X_train_res, y_train_res = smote.fit_resample(X_train_cv, y_train_cv)
    
    stacker_fold = StackingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(n_estimators=100, max_depth=4, min_samples_leaf=5, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
            ('gb', GradientBoostingClassifier(n_estimators=80, max_depth=3, learning_rate=0.05, min_samples_leaf=10, random_state=RANDOM_STATE)),
            ('et', ExtraTreesClassifier(n_estimators=100, max_depth=4, min_samples_leaf=5, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
        ],
        final_estimator=LogisticRegression(class_weight='balanced', C=0.1, max_iter=1000, random_state=RANDOM_STATE),
        cv=3,
        stack_method='predict_proba',
        n_jobs=-1
    )
    stacker_fold.fit(X_train_res, y_train_res)
    y1_cv_probas[val_idx] = stacker_fold.predict_proba(X_val_cv)[:, 1]
    print(f'Fold {fold+1}: val positives={y_val_cv.sum()}, avg proba bankrupt={y1_cv_probas[val_idx].mean():.4f}')

# Threshold tuning
print('\n--- Threshold Sweep ---')
best_thresh_1, best_acc_1 = 0.5, 0.0
for thresh in np.arange(0.02, 0.50, 0.01):
    preds = (y1_cv_probas >= thresh).astype(int)
    acc = custom_accuracy(y1, preds)
    _, rate = sparsity_check(preds)
    if rate < 0.20 and acc >= best_acc_1:
        best_acc_1 = acc
        best_thresh_1 = thresh
    if thresh in [0.05, 0.10, 0.15, 0.20, 0.30, 0.40]:
        print(f'  thresh={thresh:.2f}: acc={acc:.4f}, sparsity={rate:.4f}')

print(f'\nBest threshold: {best_thresh_1:.2f}')
y1_cv_preds = (y1_cv_probas >= best_thresh_1).astype(int)
evaluate_model(y1, y1_cv_preds, f'Cluster 1 — 5-Fold CV (thresh={best_thresh_1:.2f})')

Fold 1: val positives=7, avg proba bankrupt=0.2836
Fold 2: val positives=7, avg proba bankrupt=0.3118
Fold 3: val positives=7, avg proba bankrupt=0.2727
Fold 4: val positives=7, avg proba bankrupt=0.2780
Fold 5: val positives=7, avg proba bankrupt=0.2694

--- Threshold Sweep ---

Best threshold: 0.47
--- Cluster 1 — 5-Fold CV (thresh=0.47) ---
Custom Accuracy (TT/(TT+TF)): 0.7143
Sparsity: 0.1848 (PASS)
Confusion Matrix:
[[1137  235]
 [  10   25]]
Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.83      0.90      1372
           1       0.10      0.71      0.17        35

    accuracy                           0.83      1407
   macro avg       0.54      0.77      0.54      1407
weighted avg       0.97      0.83      0.88      1407



(np.float64(0.7142857142857143), np.float64(0.1847903340440654))

## Train Final Model & Save

In [9]:
# Train final model for cluster 1 with moderate SMOTE
smote_1 = SMOTE(random_state=RANDOM_STATE, k_neighbors=min(5, int(y1.sum()) - 1),
                sampling_strategy=min(0.3, 1.0))
X1_res, y1_res = smote_1.fit_resample(X1_sel, y1)
print(f'After SMOTE: {len(X1_res)} samples (was {len(X1_sel)})')
print(f'Class distribution: {pd.Series(y1_res).value_counts().to_dict()}')

stacker_1 = StackingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(n_estimators=100, max_depth=4, min_samples_leaf=5, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
        ('gb', GradientBoostingClassifier(n_estimators=80, max_depth=3, learning_rate=0.05, min_samples_leaf=10, random_state=RANDOM_STATE)),
        ('et', ExtraTreesClassifier(n_estimators=100, max_depth=4, min_samples_leaf=5, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
    ],
    final_estimator=LogisticRegression(class_weight='balanced', C=0.1, max_iter=1000, random_state=RANDOM_STATE),
    cv=3,
    stack_method='predict_proba',
    n_jobs=-1
)
stacker_1.fit(X1_res, y1_res)

# Training accuracy with tuned threshold
y1_train_proba = stacker_1.predict_proba(X1_sel)[:, 1]
y1_train_pred = (y1_train_proba >= best_thresh_1).astype(int)
evaluate_model(y1, y1_train_pred, f'Cluster 1 — Training (thresh={best_thresh_1:.2f})')

THRESHOLD_1 = best_thresh_1

After SMOTE: 1783 samples (was 1407)
Class distribution: {0: 1372, 1: 411}
--- Cluster 1 — Training (thresh=0.47) ---
Custom Accuracy (TT/(TT+TF)): 0.8571
Sparsity: 0.1919 (PASS)
Confusion Matrix:
[[1132  240]
 [   5   30]]
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.83      0.90      1372
           1       0.11      0.86      0.20        35

    accuracy                           0.83      1407
   macro avg       0.55      0.84      0.55      1407
weighted avg       0.97      0.83      0.88      1407



In [10]:
# Save cluster 1 model with threshold
model_1_info = {
    'model': stacker_1,
    'features': top_features_1,
    'n_features': N_FEATURES_1,
    'threshold': THRESHOLD_1,
    'cluster_id': 1,
    'model_type': 'stacking',
    'member': 'Noah Miller'
}
joblib.dump(model_1_info, os.path.join(SUBGROUP_MODEL_DIR, 'cluster_1_model.joblib'))
print(f'Cluster 1 model saved. Features: {N_FEATURES_1}, Threshold: {THRESHOLD_1:.2f}')

Cluster 1 model saved. Features: 3, Threshold: 0.47


## Summary Cluster 1

| Metric | CV (5-fold) | Training |
|---|---|---|
| Custom Accuracy (TT/(TT+TF)) | 71.43% (25/35) | 85.71% (30/35) |
| Sparsity (% predicted bankrupt) | 18.48% PASS | 19.19% PASS |
| Features (N) | 3 | 3 |
| Threshold | 0.47 | 0.47 |
| Feature Score ((50-N)/50) | 0.94 | 0.94 |

**Estimated Rank Score:** 0.3(0.857) + 0.4(0.714) + 0.3(0.94) = 0.825

**Top 3 Features:** Borrowing dependency, Equity to Liability, Total debt/Total net worth

**Model:** Stacking (RF + GB + ET → LR meta), moderate SMOTE (30%), 5-fold CV